# Committor Model: 2D Langevin Dynamics on the Z Potential, Staple Paths

Trains a non-parametric committor (`CommittorNE`) on **staple (iSTAR) path-sampling data** of a particle on the 2D Z potential with underdamped Langevin dynamics. At finite friction the committor depends on positions and velocities, q(y, x, v_y, v_x). The run records positions only, so the velocity has to come from the positions. Two setups:

- **History** (`FD_VELOCITY = False`), as `base_documented_1d_langevin.ipynb`: with lag d = 1 the basis is a polynomial in (c(t−1), c(t)) for the drawn position c. Result: `zqs_z` 30–37, does not converge (notes §10.2).
- **Finite-difference velocities** (`FD_VELOCITY = True`, the default): Δy, Δx per frame as explicit CVs, plus the ridge-normal coordinate y + x and velocity Δy + Δx (`ADD_DIAG`), fitted at lag 4 (`FIT_LAG`). Best setup so far: `zqs_z` 14 and still falling after 3000 iterations, **with forward pieces only** (`figures_by_potential/2d_zpot/langevin_g2/fd_velocity/fd_test.py`, `run_diag.log`). This notebook repeats it **with the mirrored pieces** (`REVERSED = True`), the setting that was required on the 2D Brownian Z staple run (notes §9.1).

**Path data** (read with `tis_data.py`, as `load_tis_data_cupy.py` does; see `docs/npneq_tis_notes.md`):
- **Cut at M:** each ensemble keeps only paths that cross its interface M, so the frames before that crossing are conditioned on their future. Every path is cut at its first frame past M (t\*); from there on the dynamics is unconditioned up to the path's end.
- **Mirrored pieces:** the piece before the path's last M crossing, read backwards, is a valid piece of the time-reversed dynamics (Langevin dynamics at equilibrium is reversible with v → −v), so it is added as an extra segment. Finite-difference velocities of a mirrored piece are computed backwards, so they are those of the reversed dynamics. The forward and mirrored piece of one path share a `group_traj` id, so the standard errors treat them as one unit.
- **Weights:** each accepted path counts once per Monte Carlo cycle it stays the current path (`rejections`, from tistools).

**Simulation:** `sim_istarz_2603`: staple, interfaces 0.05, 0.13, …, 0.85 on the order parameter y; Langevin with γ = 2, kT = 0.05, m = 1, dt = 0.002 (one frame per step); the Z potential `sjoelbak.RectangularGridWithBarrierPotential()` (identical to `sjoelbak_angle(Lx=0.3, angle=45)`). No exact committor is available for this run.

**Boundaries:** `λ <= 0.05` (A) and `λ >= 0.85` (B), the first and last interfaces.

## 1. Imports

In [ ]:
import argparse
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import cupy as cp

# Plot styling
plt.rc('font', size=10)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('legend', fontsize=12)

# Custom optimal-RC / committor package, and the path-data loader of the examples
REPO = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/OP_improvement/optimalrcs'
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs, metrics, plots, metrics_tis
import tis_data

# toytis, for the potential contours
TOYTIS = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/PyRETIS3/toytis'
sys.path.insert(0, TOYTIS)
from potentials.sjoelbak_angle import RectangularGridWithAngledBarrierPotential
potential = RectangularGridWithAngledBarrierPotential(Lx=0.3, angle=45)

# Figures of this notebook go to their own directory; the file names carry the run settings (TAG)
FIG_DIR = f'{REPO}/examples/figures_by_potential/2d_zpot/langevin_g2/staple_notebook'
os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(name):
    plt.gcf().savefig(f'{FIG_DIR}/{name}.png', dpi=120, bbox_inches='tight')

## 2. Data Loading

`tis_data.prepare_data` reads every ensemble folder `0NN/` (`order.txt`: time, y, x; `pathensemble.txt`: acceptance, weights, staple core indices), skips rejected cycles, cuts each path at its first frame past M (the interface the ensemble must cross: λ₀ for 001, λ₁ for 002, …), adds the mirrored pieces, and returns per-frame arrays: CVs, λ, `t_traj` (MD step), `i_traj` (piece), `group_traj` (path), `path_weights` (MC weight, mean 1), and the A/B masks.

**Parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `PATHS_PER_ENSEMBLE` | 1000 | Random subset of accepted paths per ensemble, weights kept (`None` = all; the full run is ~10⁸ frames) |
| `REVERSED` | True | Add the mirrored pieces (about doubles the frames: ~16M) |
| `WEIGHTS` | `rejections` | MC path weights |
| `FD_VELOCITY` | True | Add Δy, Δx (change over one frame, computed on the whole path before the cut) as CVs |
| `ADD_DIAG` | True | Add the ridge-normal coordinate y + x (and, with `FD_VELOCITY`, Δy + Δx) |

Training CVs: (y, x) without `FD_VELOCITY`; (y, x, Δy, Δx) with it; plus the ridge-normal columns with `ADD_DIAG`.

In [ ]:
TIS_DIR = f'{TOYTIS}/simulations/sim_istarz_2603'
PATHS_PER_ENSEMBLE = 1000
SEED = 0
REVERSED = True
WEIGHTS = 'rejections'
FD_VELOCITY = True
ADD_DIAG = True
dt = 0.002                         # MD time step = time per frame

parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
argv = [TIS_DIR, '--weights', WEIGHTS]
argv += ['--fd-velocity'] if FD_VELOCITY else ['--n-cvs', '2']     # CVs: y, x (, dy, dx)
if PATHS_PER_ENSEMBLE is not None:
    argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE), '--seed', str(SEED)]
if REVERSED:
    argv += ['--reversed']
args = parser.parse_args(argv)

data = tis_data.prepare_data(args)
lam, xpos = data.X_all[:, 0], data.X_all[:, 1]     # lambda = y (order parameter), x
X_train = data.X
if ADD_DIAG:                       # ridge-normal coordinate (the barrier runs along y + x = const) and velocity
    X_train = np.c_[X_train, X_train[:, 0] + X_train[:, 1]] if not FD_VELOCITY else \
              np.c_[X_train, X_train[:, 0] + X_train[:, 1], X_train[:, 2] + X_train[:, 3]]
lambda_a, lambda_b = args.lambda_a, args.lambda_b
n_pieces, n_paths = len(np.unique(data.i_traj)), len(np.unique(data.group_traj))
print(f'{len(lam)} frames in {n_pieces} pieces from {n_paths} paths; training CVs: {X_train.shape[1]}')

## 3. Sampled Frames

Where the cut and mirrored pieces put the frames: the weighted density of frames on (λ, x). This is the path-sampling density (more frames near the interfaces), not the equilibrium distribution, so no free energy is computed.

In [ ]:
e_y, e_x = np.linspace(-0.01, 0.91, 93), np.linspace(-0.01, 0.31, 33)
n_w = np.histogram2d(lam, xpos, [e_y, e_x], weights=data.path_weights)[0]

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
im = ax.pcolormesh(e_y, e_x, np.where(n_w > 0, n_w, np.nan).T, cmap='viridis', norm='log')
fig.colorbar(im, ax=ax, label='weighted frames')
for lb in (lambda_a, lambda_b):
    ax.axvline(lb, color='k', ls='--', lw=0.8)
ax.set(xlabel='$\\lambda$ (= y)', ylabel='x')
save_fig(f'frames_{"rev" if REVERSED else "fwd"}')
plt.show()

## 4. Train Committor (Non-Parametric)

`CommittorNE`: the path data are not an equilibrium trajectory, and time reversal flips the velocity, so the reversible estimator does not apply. Each iteration draws one training CV as `y`, with the MC path weights in the fit.

**With `FD_VELOCITY = True`: lag-Δ fit** (notes §10.1). The library's update solves Σ_t f_k(t)[q(t+1) − q(t)] = 0 over one step. With `FIT_LAG = Δ > 1` it is replaced by Σ_t w_t f_k(t)[q(min(t+Δ, τ_t)) − q(t)] = 0, with τ_t the first A/B frame or the last frame of the piece: windows stop at A/B or at the piece end, as in the stopped Z_q. Over one step underdamped dynamics nearly conserves energy, so errors that depend only on energy are almost invisible at lag 1; at lag Δ they become ~Δ times more visible. The basis is evaluated per chunk of frames (`LazyRY`), as `lazy_basis=True` does for the library update. Judge a lag-Δ fit at lags ≥ Δ.

**With `FD_VELOCITY = False`: history.** `history_type = ['y(t-d),y(t)']` draws a lag d from `history` each iteration (d = 0: basis in (r(t), y(t)); d = 1: basis in (y(t−1), y(t)), which carries that coordinate's velocity). Lag-1 fit.

Paths stop at their first A/B frame; `set_fixed_traj_length_trap` treats those ends as traps the path stays in, so the Z_q-based metrics keep the windows that run into A or B (metrics only). Convergence is judged by `zqs_z`, the worst |stopped Z_q / standard error| over q and lags: about 2 or less (notes §5, §6). For paths cut at t\* the stopped Z_q of the committor is 0 at every q and lag (notes §8.1).

**Parameters** (as `figures_by_potential/2d_zpot/langevin_g2/fd_velocity/fd_test.py`):
| Parameter | Value | Description |
|-----------|-------|-------------|
| `FIT_LAG` | 4 | Fit lag Δ in frames (`FD_VELOCITY` only; 1 = library update) |
| `history` | 0, 1 | History lag times (`FD_VELOCITY = False` only) |
| `envelope_scale` | 0.001 | RC envelope width |
| `gamma` | 0.2 | RC regularization |
| `ny` | 6 | Polynomial degree |
| `max_iter` | 3000 | Maximum iterations (the forward-only run was still improving at 3000; ~0.4 s per iteration per 8M frames) |

**Reference, forward pieces only** (same settings, `run_diag.log`): `zqs_z` 14.1; worst |z| at lags 1, 2, 4, 8, 16, 64, 256, 1024: 2.3 1.3 2.1 3.7 6.8 14.1 10.7 8.2.

**Seeds and warm-up** (`seed_runs.run_seeds`, shared with the other notebooks): the fit is repeated for every seed in `SEEDS` (the seed sets which CV and envelope centre each iteration draws). The first `WARMUP_ITER` iterations of every seed can use their own envelope (`WARMUP_ENVELOPE`), gamma (`WARMUP_GAMMA`) and CV (`WARMUP_CV`: `None` random, otherwise the index of the CV to use); then `ENVELOPE` and `GAMMA` take over (`WARMUP_ITER = 0`: no warm-up). After the warm-up and every `CHUNK` iterations the plots in `PLOTS` are shown here and saved to `<OUT>/seed<s>/it<N>_<plot>.png`, with a progress line. The objects of earlier seeds are freed; `q` is the last seed's fit and `results[seed]['r']` every seed's RC.

**Committor maps per chunk** (`seed_runs.committor_map`): after the warm-up and every chunk, the mean q per bin in the (λ, x) plane on the effective potential and in the (λ, v_λ) and (λ, v_x) planes, as in section 6 (velocities estimated from the positions; no exact committor for this run); saved as `it<N>_q_planes.png`.

In [ ]:
FIT_LAG = 4
MAX_ITER = 3000                    # iterations per seed (after the warm-up)
SEEDS = [0]
CHUNK = 1000                       # iterations between shown and saved plots
WARMUP_ITER = 0                    # optional warm-up (0 = off)
WARMUP_ENVELOPE = 0.001
WARMUP_GAMMA = 0.2
WARMUP_CV = None                   # None: random CV; otherwise the column of X_train to use
PLOTS = ('metrics', 'tis')         # add 'feps', 'obs_pred' if GPU memory allows (~16M frames)
TAG = f"{'rev' if REVERSED else 'fwd'}_{'fd' if FD_VELOCITY else 'hist'}{'_diag' if ADD_DIAG else ''}_lag{FIT_LAG if FD_VELOCITY else 1}"   # run settings, in the figure names
ENVELOPE = 0.001                   # envelope_scale
history = [0, 1]                   # lag times (history setup only)
GAMMA = 0.2                        # regularization strength
yr_type = ['y(t-d),y(t)']          # c(t-d) and c(t) for the drawn position c
X_gpu = cp.asarray(X_train)
b = data.boundary0 | data.boundary1
tau = metrics_tis.stop_index(data.i_traj, b)   # first A/B frame or the piece's last frame


class LazyRY:
    """basis_poly_ry evaluated on chunks of frames, normalized over all frames as basis_poly_ry does."""
    def __init__(self, r, y, n, fenv):
        self.r, self.y, self.n = r / cp.max(cp.abs(r)), y / cp.max(cp.abs(y)), n
        self.fenv = cp.ones_like(r) if fenv is None else fenv
    def cols(self, idx):
        return optimalrcs._poly_ry(self.r[idx], self.y[idx], self.n, self.fenv[idx])


if not hasattr(optimalrcs.nonparametrics, 'npneq_lag1'):
    optimalrcs.nonparametrics.npneq_lag1 = optimalrcs.nonparametrics.npneq     # keep the library's update
optimalrcs.nonparametrics.npneq = optimalrcs.nonparametrics.npneq_lag1
fit_kw = dict(lazy_basis=True)
if not FD_VELOCITY:
    fit_kw.update(history_delta_t=history, history_type=yr_type, history_shift_type='r(t)')
elif FIT_LAG > 1:
    n = len(lam); tau_np = cp.asnumpy(tau)
    u_np = np.minimum(np.arange(n) + FIT_LAG, tau_np)
    U = cp.asarray(u_np.astype(np.int32))
    STARTS = cp.asarray(np.flatnonzero(~b & (u_np > np.arange(n))).astype(np.int32))
    W = cp.asarray(data.path_weights)
    del u_np, tau_np

    def npneq_lag(r_traj, fk, i_traj=None, gamma=0, stable=False, train_mask=None, chunk=None, w_traj=None, **kw):
        akj, bb = 0, 0
        for s0 in range(0, STARTS.size, 400_000):
            st = STARTS[s0:s0 + 400_000]
            fa, fb, wt = fk.cols(st), fk.cols(U[st]), W[st]
            akj = akj + fa @ (fb * wt - fa * (wt + gamma)).T
            bb = bb + fa @ (-(r_traj[U[st]] - r_traj[st]) * wt)
        al = cp.linalg.lstsq(akj, bb, rcond=None)[0]
        new = r_traj.copy()
        for s0 in range(0, n, 1_000_000):
            idx = cp.arange(s0, min(s0 + 1_000_000, n))
            new[idx] = r_traj[idx] + al @ fk.cols(idx)
        return cp.clip(new, 0, 1)
    optimalrcs.nonparametrics.npneq = npneq_lag
    fit_kw = dict(basis_functions=lambda r_, y_, n_, f_: LazyRY(r_, y_, n_, f_))

import seed_runs

def make_q():
    q = optimalrcs.CommittorNE(
        boundary0=data.boundary0,      # state A
        boundary1=data.boundary1,      # state B
        i_traj=data.i_traj,            # piece (forward or mirrored)
        t_traj=data.t_traj,            # MD step
        path_weights=data.path_weights,
        group_traj=data.group_traj,    # path: its two pieces are not independent
    )
    q.set_fixed_traj_length_trap(b, 2 ** 16)
    return q

# Worst |stopped Z_q / standard error| per lag, to compare with the forward-only run (markdown above)
LAGS = (1, 2, 4, 8, 16, 64, 256, 1024)
def report(q):
    zs = []
    for lag in LAGS:
        _, z = metrics_tis.comp_zq_stopped_zscore(q.r_traj, b, data.i_traj, data.path_weights, lag, tau=tau,
                                                  g_traj=data.group_traj)
        zs.append(float(cp.max(cp.abs(z[5:]))))
    return f'| worst |z| at lags {LAGS}: ' + ' '.join(f'{z:.1f}' for z in zs)

# After every chunk: mean q per bin in the (lambda, x) plane on the effective potential, and in the
# (lambda, v_lambda) and (lambda, v_x) planes, with the velocities estimated from the positions
# (section 6; v_lam, v_x, gy, gx, V and lev are used in sections 5 and 6 as well)
def fd_velocity(c):
    v = np.gradient(c) / dt                     # central differences, one-sided at the ends
    same_prev = np.r_[False, data.i_traj[1:] == data.i_traj[:-1]]
    same_next = np.r_[data.i_traj[1:] == data.i_traj[:-1], False]
    fwd = np.r_[c[1:] - c[:-1], 0] / dt         # piece edges: one-sided within the piece
    bwd = np.r_[0, c[1:] - c[:-1]] / dt
    v = np.where(same_prev & same_next, v, np.where(same_next, fwd, bwd))
    return v

v_lam, v_x = fd_velocity(lam), fd_velocity(xpos)
v_max = 4 * np.sqrt(0.05)          # 4 thermal velocities (kT = 0.05, m = 1)
gy, gx = np.linspace(-0.01, 0.91, 185), np.linspace(-0.01, 0.31, 65)
V = np.array([[potential.effective_potential((np.array([yv, xv]), np.zeros(2))) for xv in gx] for yv in gy])
lev = np.linspace(np.percentile(V, 1), np.percentile(V, 92), 14)
e_lam = np.linspace(lambda_a, lambda_b, 81)
chunk_map = seed_runs.committor_map([
    seed_runs.plane(lam, e_lam, '$\\lambda$ (= y)', xpos, np.linspace(-0.01, 0.31, 33), 'x', contours=(gy, gx, V, lev)),
    seed_runs.plane(lam, e_lam, '$\\lambda$ (= y)', v_lam, np.linspace(-v_max, v_max, 41), '$v_\\lambda$'),
    seed_runs.plane(lam, e_lam, '$\\lambda$ (= y)', v_x, np.linspace(-v_max, v_max, 41), '$v_x$'),
], mask=~b)

results, q = seed_runs.run_seeds(
    optimalrcs, make_q,
    comp_y=lambda: X_gpu[:, np.random.randint(X_gpu.shape[1])],
    fit_kw=dict(ny=6, print_step=250, save_min_metric='max_z_zq_stopped',
                metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'),
                **fit_kw),
    seeds=SEEDS, max_iter=MAX_ITER, chunk=CHUNK, out_dir=f'{FIG_DIR}/seeds_{TAG}', envelope=ENVELOPE, gamma=GAMMA,
    warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=WARMUP_GAMMA,
    warmup_comp_y=None if WARMUP_CV is None else (lambda: X_gpu[:, WARMUP_CV]),
    plots=PLOTS, plot_options=dict(metrics=['delta_r2', 'max_z_zq_stopped', 'delta_x']),
    extra_plot={'q_planes': chunk_map},
    report=report,
)

## 5. Committor against λ and on the Potential

q of every frame against the order parameter, and the mean q per (λ, x) bin on the potential (grey: contours of the effective potential including the constant force along y; dashed: q = 0.1, 0.5, 0.9). At a given λ between the ridge and the plateau, frames on either side of the 45° barrier have very different q, so λ alone is not a good RC.

In [ ]:
r = cp.asnumpy(q.r_traj)           # gy, gx, V and lev (the effective potential): computed in section 4
free = ~(data.boundary0 | data.boundary1)

def q_map(other, edges, mask=free):
    e_lam = np.linspace(lambda_a, lambda_b, 81)
    n = np.histogram2d(lam[mask], other[mask], [e_lam, edges])[0]
    s = np.histogram2d(lam[mask], other[mask], [e_lam, edges], weights=r[mask])[0]
    return e_lam, np.where(n > 20, s / np.maximum(n, 1), np.nan)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={'width_ratios': [1, 1.6]})
show = np.random.default_rng(0).permutation(len(r))[:300_000]
ax1.scatter(lam[show], r[show], s=1, alpha=0.2, edgecolors='none')
for lb in (lambda_a, lambda_b):
    ax1.axvline(lb, color='k', ls='--', lw=0.5)
ax1.set(xlabel='$\\lambda$', ylabel='$q$')

e_lam, m = q_map(xpos, np.linspace(-0.01, 0.31, 33))
e_x = np.linspace(-0.01, 0.31, 33)
im = ax2.pcolormesh(e_lam, e_x, m.T, cmap='coolwarm', vmin=0, vmax=1)
ax2.contour(gy, gx, V.T, levels=lev, colors='0.4', linewidths=0.5)
ax2.contour((e_lam[1:] + e_lam[:-1]) / 2, (e_x[1:] + e_x[:-1]) / 2, np.nan_to_num(m, nan=0.5).T,
            levels=[0.1, 0.5, 0.9], colors='k', linewidths=1, linestyles='--')
ax2.set(xlabel='$\\lambda$ (= y)', ylabel='x', xlim=(lambda_a, lambda_b), ylim=(-0.01, 0.31))
fig.colorbar(im, ax=ax2, label='$q$')
plt.tight_layout()
save_fig(f'committor_{TAG}')
plt.show()

## 6. Committor in the (λ, x), (λ, v_λ) and (λ, v_x) Planes

The run does not record velocities, so for this check they are estimated from the positions: the central difference (c(t+1) − c(t−1)) / 2dt within each piece (one-sided at piece ends). With underdamped Langevin dynamics the positions move smoothly over one step, so this is close to the true velocity. In mirrored pieces the time runs backwards, so their velocities are already those of the reversed dynamics (−v), consistent with how the pieces enter the fit. With `FD_VELOCITY = False` these velocities are not used in training; with it, training uses the one-sided change over one frame, which is close to this.
- (λ, v_λ): if the history captured the velocity, q depends on v_λ and not only on λ: particles moving towards B have higher q.
- (λ, v_x): the barrier runs at 45°, so near it the transverse velocity matters as well.

In [ ]:
# v_lam, v_x (central differences within each piece, fd_velocity) and v_max: computed in section 4
planes = [(xpos, np.linspace(-0.01, 0.31, 33), 'x'),
          (v_lam, np.linspace(-v_max, v_max, 41), '$v_\\lambda$'),
          (v_x, np.linspace(-v_max, v_max, 41), '$v_x$')]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (other, edges, label) in zip(axes, planes):
    e_lam, m = q_map(other, edges)
    im = ax.pcolormesh(e_lam, edges, m.T, cmap='coolwarm', vmin=0, vmax=1)
    ax.contour((e_lam[1:] + e_lam[:-1]) / 2, (edges[1:] + edges[:-1]) / 2, np.nan_to_num(m, nan=0.5).T,
               levels=[0.1, 0.5, 0.9], colors='k', linewidths=1, linestyles='--')
    if label == 'x':
        ax.contour(gy, gx, V.T, levels=lev, colors='0.4', linewidths=0.5)
        ax.set_ylim(-0.01, 0.31)
    ax.set(xlabel='$\\lambda$ (= y)', ylabel=label, xlim=(lambda_a, lambda_b))
    fig.colorbar(im, ax=ax, label='$q$')
plt.tight_layout()
save_fig(f'planes_{TAG}')
plt.show()

## 7. Save Results

Uncomment to save the committor with the frames (positions, estimated velocities, piece, path, weight).

In [ ]:
# np.savez('q_staple_zlangevin.npz', q=cp.asnumpy(q.r_traj), lam=lam, x=xpos, v_lam=v_lam, v_x=v_x,
#          i_traj=data.i_traj, group_traj=data.group_traj, t_traj=data.t_traj,
#          path_weights=data.path_weights, lambda_a=lambda_a, lambda_b=lambda_b)